In [33]:


import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

In [34]:


df = pd.read_csv("ethereum_data.csv")

print(df.head())
print(df.shape)

   block_number                                   transaction_hash  \
0         46147  0x5c504ed432cb51138bcf09aa5e8a410dd4a1e204ef84...   
1         46169  0x19f1df2c7ee6b464720ad28e903aeda1a5ad8780afc2...   
2         46170  0x9e6e19637bb625a8ff3d052b7c2fe57dc78c55a15d25...   
3         46194  0xcb9378977089c773c074045b20ede2cdcc3a6ff562f4...   
4         46205  0x570ce19176bd0002b04a9179309129bbdaf0c4252ffe...   

                                         from  \
0  0xA1E4380A3B1f749673E270229993eE55F35663b4   
1  0xbD08e0cDDEc097DB7901EA819a3d1FD9de8951A2   
2  0x63Ac545C991243fa18aec41D4F6f598e555015dc   
3  0x037dd056e7FDBd641DB5b6BEA2A8780A83FAe180   
4  0x3f2F381491797Cc5C0D48296C14Fd0Cd00Cdfa2D   

                                           to                  value  
0  0x5DF9B87991262F6BA471F09758CDE1c0FC1De734                  31337  
1  0x5C12A8e43Faf884521C2454f39560e6C265a68C8   19900000000000000000  
2  0xC93f2250589a6563f5359051c1eA25746549f0D8  599989500000000000000  


In [35]:


df.columns = df.columns.str.lower().str.strip()

df["from"] = df["from"].astype(str).str.lower().str.strip()
df["to"] = df["to"].astype(str).str.lower().str.strip()

df["value"] = pd.to_numeric(df["value"], errors="coerce")

df.dropna(inplace=True)
df.drop_duplicates(inplace=True)

print(df.shape)

(1665020, 5)


In [36]:


recv = df.groupby("to").agg(
    recv_tx=("transaction_hash", "count"),
    recv_total=("value", "sum"),
    recv_avg=("value", "mean"),
    unique_senders=("from", "nunique")
).reset_index()

recv.rename(columns={"to": "wallet"}, inplace=True)

recv.head()

,wallet,recv_tx,recv_total,recv_avg,unique_senders
0,0x0000000000000000000000000000000000000000,193,4.483329e+21,2.322968e+19,99
1,0x0000000000000000000000000000000000000001,1,0.000000e+00,0.000000e+00,1
2,0x0000000000000000000000000000000000000027,1,6.616908e+15,6.616908e+15,1
3,0x0000000000000000000000000000000000000123,1,3.500000e+01,3.500000e+01,1
4,0x00000000742a16c2ccfdf3a6fde782e6e95b4120,1,9.884823e+15,9.884823e+15,1


In [37]:


sent = df.groupby("from").agg(
    sent_tx=("transaction_hash", "count"),
    sent_total=("value", "sum"),
    sent_avg=("value", "mean"),
    unique_receivers=("to", "nunique")
).reset_index()

sent.rename(columns={"from": "wallet"}, inplace=True)

sent.head()

,wallet,sent_tx,sent_total,sent_avg,unique_receivers
0,0x0000000103026f36d9f2ba6468d2816cd5dce83a,1,9.500000e+16,9.500000e+16,1
1,0x00006314ee6ba5a9421e4aa6a47c6867a882bd92,2,1.060000e+18,5.300000e+17,2
2,0x000083ceb2317f5755be7a745e3c4be7ba396877,3,1.000000e+19,3.333333e+18,1
3,0x0001be2782b76273093874aa372870ce6e418b22,1,1.008950e+18,1.008950e+18,1
4,0x000251104ce432bd728a5712f175eff4e446023f,1,9.000000e+17,9.000000e+17,1


In [38]:


wallet = pd.merge(recv, sent, on="wallet", how="outer").fillna(0)

wallet.head()

,wallet,recv_tx,recv_total,recv_avg,unique_senders,sent_tx,sent_total,sent_avg,unique_receivers
0,0x0000000000000000000000000000000000000000,193.0,4.483329e+21,2.322968e+19,99.0,0.0,0.0,0.0,0.0
1,0x0000000000000000000000000000000000000001,1.0,0.000000e+00,0.000000e+00,1.0,0.0,0.0,0.0,0.0
2,0x0000000000000000000000000000000000000027,1.0,6.616908e+15,6.616908e+15,1.0,0.0,0.0,0.0,0.0
3,0x0000000000000000000000000000000000000123,1.0,3.500000e+01,3.500000e+01,1.0,0.0,0.0,0.0,0.0
4,0x00000000742a16c2ccfdf3a6fde782e6e95b4120,1.0,9.884823e+15,9.884823e+15,1.0,0.0,0.0,0.0,0.0


In [39]:


wallet["flow_ratio"] = wallet["sent_total"] / (wallet["recv_total"] + 1)

wallet["activity_score"] = wallet["recv_tx"] + wallet["sent_tx"]

wallet["network_score"] = wallet["unique_senders"] + wallet["unique_receivers"]

wallet.head()

,wallet,recv_tx,recv_total,recv_avg,unique_senders,sent_tx,sent_total,sent_avg,unique_receivers,flow_ratio,activity_score,network_score
0,0x0000000000000000000000000000000000000000,193.0,4.483329e+21,2.322968e+19,99.0,0.0,0.0,0.0,0.0,0.0,193.0,99.0
1,0x0000000000000000000000000000000000000001,1.0,0.000000e+00,0.000000e+00,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
2,0x0000000000000000000000000000000000000027,1.0,6.616908e+15,6.616908e+15,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
3,0x0000000000000000000000000000000000000123,1.0,3.500000e+01,3.500000e+01,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
4,0x00000000742a16c2ccfdf3a6fde782e6e95b4120,1.0,9.884823e+15,9.884823e+15,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0


In [40]:


wallet["label"] = (
    (wallet["unique_senders"] > 15) &
    (wallet["unique_receivers"] > 10) &
    (wallet["activity_score"] > 25)
).astype(int)

print(wallet["label"].value_counts())

label
0    55325
1       25
Name: count, dtype: int64


In [41]:


X = wallet[
    [
        "recv_tx",
        "recv_total",
        "recv_avg",
        "unique_senders",
        "sent_tx",
        "sent_total",
        "sent_avg",
        "unique_receivers",
        "flow_ratio",
        "activity_score",
        "network_score"
    ]
]

y = wallet["label"]

In [42]:


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [43]:


model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)

print("Model Trained Successfully")

Model Trained Successfully


In [44]:
pred = model.predict(X_test)

acc = accuracy_score(y_test, pred)

print("Accuracy:", acc)
print("Accuracy Percentage: {:.2f}%".format(acc * 100))

print(classification_report(y_test, pred))

Accuracy: 1.0
Accuracy Percentage: 100.00%
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     11065
           1       1.00      1.00      1.00         5

    accuracy                           1.00     11070
   macro avg       1.00      1.00      1.00     11070
weighted avg       1.00      1.00      1.00     11070



In [45]:

def check_wallet(address):

    address = address.lower().strip()

    row = wallet[wallet["wallet"] == address]

    if len(row) == 0:
        print("Wallet not found in dataset.")
        return

    X_new = row[
        [
            "recv_tx",
            "recv_total",
            "recv_avg",
            "unique_senders",
            "sent_tx",
            "sent_total",
            "sent_avg",
            "unique_receivers",
            "flow_ratio",
            "activity_score",
            "network_score"
        ]
    ]

    pred = model.predict(X_new)[0]
    prob = model.predict_proba(X_new)[0][1]

    if pred == 1:
        risk = "HIGH RISK"
    elif prob > 0.35:
        risk = "MEDIUM RISK"
    else:
        risk = "LOW RISK"

    print("\nWallet Address:", address)
    print("Risk Level:", risk)
    print("Confidence:", round(prob * 100, 2), "%")

In [46]:


X_all = wallet[
    [
        "recv_tx",
        "recv_total",
        "recv_avg",
        "unique_senders",
        "sent_tx",
        "sent_total",
        "sent_avg",
        "unique_receivers",
        "flow_ratio",
        "activity_score",
        "network_score"
    ]
]

wallet["risk_score"] = model.predict_proba(X_all)[:,1]

top10 = wallet.sort_values("risk_score", ascending=False).head(10)

top10[
    [
        "wallet",
        "risk_score",
        "recv_tx",
        "sent_tx",
        "unique_senders",
        "unique_receivers",
        "activity_score"
    ]
]

,wallet,risk_score,recv_tx,sent_tx,unique_senders,unique_receivers,activity_score
54434,0xfbb1b73c4f0bda4f67dca266ce6ef42f520fbb98,0.996667,46677.0,4338.0,1389.0,1754.0,51015.0
4392,0x120a270bbc009644e35f0bb6ab13f95b8199c4ad,0.980000,4274.0,12555.0,3541.0,4023.0,16829.0
24107,0x6e4f24a297c496aef5d404725a9bbd4cc23ac795,0.976667,470.0,80.0,114.0,56.0,550.0
1663,0x056507108fe6d30189eccd6cd7f4fad56eeef622,0.973333,211.0,640.0,33.0,104.0,851.0
11292,0x32be343b94f860124dc4fee278fdcbd38c102d88,0.970000,348870.0,17387.0,6062.0,5318.0,366257.0
22665,0x6747f922385c9a6f9cfdf095f34345257dd597be,0.970000,217.0,167.0,190.0,159.0,384.0
23244,0x69ea6b31ef305d6b99bb2d4c9d99456fa108b02a,0.970000,290.0,135.0,95.0,75.0,425.0
33643,0x9b0a028eafdecde3afc0fd00b7937098388b7c8a,0.970000,9763.0,704.0,454.0,310.0,10467.0
9219,0x2910543af39aba0cd09dbb2d50200b3e800a63d2,0.966667,4580.0,8553.0,1844.0,2903.0,13133.0
51238,0xed6b25b3b2dab2d5d96ac659aabbf812f069351b,0.960000,1514.0,347.0,625.0,225.0,1861.0


In [47]:
user_wallet = input("Enter Ethereum Wallet Address: ").strip().lower()

if user_wallet.startswith("0x") and len(user_wallet) == 42:
    check_wallet(user_wallet)
else:
    print("Invalid Ethereum wallet address.")


Wallet Address: 0xfbb1b73c4f0bda4f67dca266ce6ef42f520fbb98
Risk Level: HIGH RISK
Confidence: 99.67 %
